# Практическая работа № 3. Сборка пайплайна предобработки

**Выполнил:** Айдар Нурбекович

**Цель:** собрать пайплайн обработки данных без утечки информации из тестовой выборки.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
paths = [
    Path("../PR1/dataset.csv"),
    Path("PR1/dataset.csv")
]

dataset_path = next((p for p in paths if p.is_file()), None)

if dataset_path is None:
    raise FileNotFoundError("Не найден Spotify Dataset")

df = pd.read_csv(dataset_path)

print("Размер датасета:", df.shape)
display(df.head())

Размер датасета: (114000, 21)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [3]:
numeric_features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "duration_ms"
]

categorical_features = [
    "track_genre"
]

target = "popularity"

selected_columns = numeric_features + categorical_features + [target]

data = df[selected_columns].copy()

print("Размер выбранных данных:", data.shape)
print("\nПропуски:")
print(data.isnull().sum())

Размер выбранных данных: (114000, 12)

Пропуски:
danceability        0
energy              0
loudness            0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
duration_ms         0
track_genre         0
popularity          0
dtype: int64


In [4]:
data = data.dropna(subset=[target])

X = data[numeric_features + categorical_features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (91200, 11)
Test: (22800, 11)


In [5]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [7]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="unknown"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print(preprocessor) 

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['danceability', 'energy', 'loudness',
                                  'speechiness', 'acousticness',
                                  'instrumentalness', 'liveness', 'valence',
                                  'tempo', 'duration_ms']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='unknown',
                                                                strategy='constant')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                               

In [9]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Train после обработки:", X_train_processed.shape)
print("Test после обработки:", X_test_processed.shape)

Train после обработки: (91200, 124)
Test после обработки: (22800, 124)


In [11]:
from scipy import sparse


assert X_train_processed.shape[1] == X_test_processed.shape[1]

def check_finite(matrix):
    if sparse.issparse(matrix):
        return np.isfinite(matrix.data).all()
    return np.isfinite(matrix).all()

assert check_finite(X_train_processed), "Ошибка в Train!"
assert check_finite(X_test_processed), "Ошибка в Test!"

print("Проверка пройдена!")
print("Train:", X_train_processed.shape)
print("Test:", X_test_processed.shape)
print("Количество итоговых признаков:", X_train_processed.shape[1])

Проверка пройдена!
Train: (91200, 124)
Test: (22800, 124)
Количество итоговых признаков: 124


In [12]:
from scipy import sparse

assert X_train_processed.shape[1] == X_test_processed.shape[1]

def check_finite(matrix):
    values = matrix.data if sparse.issparse(matrix) else matrix
    return np.isfinite(values).all()

assert check_finite(X_train_processed)
assert check_finite(X_test_processed)

print("Проверка пройдена!")
print("Train:", X_train_processed.shape)
print("Test:", X_test_processed.shape)

Проверка пройдена!
Train: (91200, 124)
Test: (22800, 124)


## Вывод

В этой работе я подготовил Spotify Dataset для машинного обучения.

- Разделил данные на 80% train и 20% test.
- Для числовых признаков использовал заполнение пропусков медианой и StandardScaler.
- Для музыкальных жанров применил OneHotEncoder.
- Объединил обработку через ColumnTransformer и Pipeline.

**Проверка утечки данных:** предобработка обучалась только на train. К test применялись уже полученные преобразования.

После обработки количество признаков в train и test совпало.